# Reproduce the PESO replication in Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hong-yiii/cp4285_reccomender_systems/blob/main/notebooks/03_peso_colab.ipynb)

This notebook re-evaluates our **trained seed-43 weights** for PESO (Yoo et al., ICLR 2026) and its two baselines. It uses PESO's own evaluation script, unchanged. Run it top to bottom and it prints the Table 2 comparison: our recorded numbers, the numbers you just reproduced, and the paper's. It does not retrain: training takes about 9 GPU-hours per seed (see `docs/PESO-REPLICATION.md`).

**Before you start**
1. **Runtime → Change runtime type → GPU: A100 or L4.** The model runs in bfloat16, which a T4 cannot run efficiently.
2. **Hugging Face access, for Meta's base model only:**
   - accept Meta's licence on [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B);
   - add a read token as a Colab secret named `HF_TOKEN` (the key icon in the left sidebar).

   Our trained weights download without login from this repository's GitHub release (`llama-peso-weights-seed43-v1`, 9.7 GB, Llama-3.2-1B-PESO-replication).
3. **Expected time:** the paper's average (blocks 1–3, 5,882 test users per method) takes about 30–60 minutes. Set `FULL = True` for all five blocks (25,166 users per method).

Built with Llama. The weights are a Llama 3.2 derivative, distributed under the Llama 3.2 Community License (`LICENSE.txt` and `NOTICE.txt` in the release).

In [ ]:
# Settings: change only these.
SEED = 43          # the bundled seed (43 ran on Modal A100s)
FULL = False       # False: blocks 1-3, the paper's average; True: all five blocks
RELEASE = "https://github.com/Hong-yiii/cp4285_reccomender_systems/releases/download/llama-peso-weights-seed43-v1"
UPSTREAM_COMMIT = "edbec5f68ddc83e39f22a041f1f2eaf1c4ad172e"
BASE_REVISION = "4e20de362430cd3b72f300e6b0f18e50e7166e08"

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU: switch the runtime to A100 or L4")

## 1. Install the replication's pinned versions

torch 2.5.1 and transformers 4.52.4 match the training runs. Run this cell before anything imports torch; if Colab already imported a different torch, restart the session and rerun from here.

In [ ]:
%pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q transformers==4.52.4 tokenizers==0.21.1 safetensors==0.5.3 huggingface_hub==0.32.3 accelerate==1.7.0
%pip install -q scipy  # PESO's continual_lora.py imports it; Colab has it, a bare Python does not
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 2. Fetch PESO's code and data, the base model and our trained weights

PESO's repository has no licence file, so it is cloned from its authors at the pinned commit rather than copied. It contains the released Instruments blocks and semantic-ID index. The base model comes from Meta's gated Hugging Face repo, and our weights from this repository's GitHub release. Every release file is checked against its SHA-256, and a rerun skips files that are already verified.

In [ ]:
import hashlib, http.client, json, os, shutil, urllib.request
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in the environment"

work = Path("/content" if Path("/content").exists() else Path.cwd() / "peso-colab")
work.mkdir(parents=True, exist_ok=True)
upstream = work / "peso"
if not upstream.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/hsyoo32/peso", str(upstream)], check=True)
subprocess.run(["git", "-C", str(upstream), "checkout", "-q", UPSTREAM_COMMIT], check=True)

from huggingface_hub import snapshot_download
base = snapshot_download("meta-llama/Llama-3.2-1B", revision=BASE_REVISION,
                         allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def download(asset, target, attempts=3):
    for _ in range(attempts):
        try:
            with urllib.request.urlopen(f"{RELEASE}/{asset}", timeout=60) as response, open(target, "wb") as f:
                shutil.copyfileobj(response, f, length=1 << 24)
            return
        except (OSError, http.client.HTTPException) as error:  # dropped connection: start the file again
            print(f"retrying {asset}: {error}")
    raise RuntimeError(f"Could not download {RELEASE}/{asset}")

assets = work / "release"
assets.mkdir(exist_ok=True)
download("MANIFEST.json", assets / "MANIFEST.json")
manifest = json.loads((assets / "MANIFEST.json").read_text())
for asset, meta in manifest["assets"].items():
    target = assets / asset
    if not (target.exists() and sha256(target) == meta["sha256"]):
        print(f"downloading {asset} ...", flush=True)
        download(asset, target)
        assert sha256(target) == meta["sha256"], f"Checksum mismatch: {asset}; rerun this cell"
    print(f"verified {asset} ({meta['bytes'] / 1e9:.2f} GB)")

## 3. Rebuild the checkpoint layout PESO's evaluator expects

The release holds only what training changed: the LoRA adapters, with every block's snapshot, and the trained embedding and output layers. PESO's loader builds the frozen base model first and then loads these files with `strict=False`, so each block's state is restored exactly.

Single LoRA and PESO both start from the pretraining stage's block 0, so their block-0 files are rebuilt from it instead of being downloaded twice.

In [ ]:
from safetensors.torch import load_file, save_file

TAGS = {
    "pretrain": ("QUICK_ws20.msl-1.temp0.8.sflora_pretrain.blocks5", "lora_pretrain"),
    "lora": ("QUICK_ws20.msl-1.temp0.8.sflora_cl_0.0.blocks5", "lora"),
    "peso": ("QUICK_ws20.msl-1.temp0.8.sflora_kldiv_latest_cl_2.0.blocks5", "lora_kldiv_latest"),
}
ckpt = work / "ckpt"
shared = [a for a, m in manifest["assets"].items() if "/" not in m["path"]]  # tokenizer, config
for asset, meta in manifest["assets"].items():
    method, _, rest = meta["path"].partition("/")
    if method in TAGS:
        target = ckpt / TAGS[method][0] / rest
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(assets / asset, target)
for method, (tag, _) in TAGS.items():
    for asset in shared:
        shutil.copy2(assets / asset, ckpt / tag / manifest["assets"][asset]["path"])
pretrain_block0 = load_file(ckpt / TAGS["pretrain"][0] / "block_0" / "model.safetensors")
subset = {k: v for k, v in pretrain_block0.items() if ".modules_to_save.default." in k}
for path in manifest["derived"]:
    method, _, rest = path.partition("/")
    target = ckpt / TAGS[method][0] / rest
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        save_file(subset, target)
print({tag: sorted(p.name for p in (ckpt / tag).iterdir()) for tag, _ in TAGS.values()})

## 4. Evaluate with PESO's own `continual_test.py`

These are the same settings as the replication: 10 beams, one user at a time, with each block evaluated on the model as it stood after training that block. Batching users is not an option: it changed almost every top-10 list in our tests. The three methods run in parallel on GPUs with at least 40 GB, otherwise one after another (about twice as long).

In [ ]:
COMMON = [
    "--dataset", "Instruments_cl_ori", "--data_path", "../data",
    "--index_file", ".index.epoch20000_edim32_beta0.0.json",
    "--temperature", "0.8", "--max_seq_len", "-1", "--max_his_len", "20", "--num_blocks", "5",
    "--lora_target_modules", "q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj",
    "--lora_modules_to_save", "embed_tokens,lm_head",
]
blocks = range(5) if FULL else (1, 2, 3)
results = work / "results"
results.mkdir(exist_ok=True)
env = {**os.environ, "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"}

def evaluate(method, port):
    tag, flag = TAGS[method]
    for block in blocks:
        out = results / f"{method}.block{block}.json"
        if out.exists():
            continue
        cmd = ["torchrun", "--nproc_per_node=1", f"--master_port={port}", "continual_test.py",
               "--ckpt_path", str(ckpt / tag), "--base_model", base, "--seed", str(SEED),
               "--results_file", str(out), "--test_batch_size", "1", "--num_beams", "10",
               "--shift_flag", flag, "--test_block", str(block), *COMMON]
        if method != "pretrain":
            cmd.append("--use_final_model")
        log = results / f"{method}.log"
        print(f"{method}: evaluating block {block} ...", flush=True)
        with log.open("a") as f:
            code = subprocess.run(cmd, cwd=upstream / "Fine-tuning", env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        if code:
            raise RuntimeError(f"{method} block {block} failed; end of {log}:\n{log.read_text()[-3000:]}")

# Each evaluator holds about 7.6 GB; three fit on a 40/80 GB A100, one at a time on a 24 GB L4.
import torch
from concurrent.futures import ThreadPoolExecutor
workers = 3 if torch.cuda.get_device_properties(0).total_memory >= 40e9 else 1
with ThreadPoolExecutor(workers) as pool:
    list(pool.map(evaluate, TAGS, (29600, 29601, 29602)))
print("done:", sorted(p.name for p in results.glob("*.json")))

## 5. Compare with our recorded run and the paper

"Recorded" values are the seed-43 numbers from the original run on Modal A100s, in `replication/peso/results/`. Expect close but not always identical numbers.

- **GPU arithmetic matters.** Upstream's evaluator on an RTX 4090 changed the outcome for 1 (single LoRA) and 4 (PESO) of the 1,772 block-1 users of seed 42, compared with the A100. That is about ±0.0002–0.0005 NDCG@10, the same size as the measured PESO − single LoRA differences.
- **Match the hardware.** Use an A100 for the closest reproduction. The paper's numbers are Table 2 (Instruments, averaged over D2–D4), from arXiv 2510.25093v3, p. 8.

In [ ]:
RECORDED = {  # seed 43, NDCG@10 and Hit@10 per block (replication/peso/results/seed43)
    "pretrain": {0: (0.015181, 0.025519), 1: (0.016034, 0.028217), 2: (0.014613, 0.024163), 3: (0.007702, 0.014417), 4: (0.010670, 0.018320)},
    "lora":     {0: (0.015181, 0.025519), 1: (0.016060, 0.029910), 2: (0.014757, 0.021417), 3: (0.014894, 0.026649), 4: (0.011725, 0.021895)},
    "peso":     {0: (0.015181, 0.025519), 1: (0.016758, 0.032167), 2: (0.014564, 0.022515), 3: (0.013276, 0.024465), 4: (0.012243, 0.022788)},
}
PAPER = {"pretrain": (0.0131, 0.0216), "lora": (0.0150, 0.0253), "peso": (0.0162, 0.0268)}

def mean(values):
    return sum(values) / len(values)

print("| Method | Block | NDCG@10 now | recorded | Hit@10 now | recorded |")
print("| --- | --- | --- | --- | --- | --- |")
now = {}
for method in TAGS:
    for block in blocks:
        r = json.loads((results / f"{method}.block{block}.json").read_text())
        m = r["all_block_results"][f"block_{block}"]["all_prompt_results"][0]
        now[method, block] = (m["ndcg@10"], m["hit@10"])
        rec = RECORDED[method][block]
        print(f"| {method} | {block} | {m['ndcg@10']:.4f} | {rec[0]:.4f} | {m['hit@10']:.4f} | {rec[1]:.4f} |")

print("\nAverage over blocks 1-3 (the paper's D2-D4):")
print("| Method | NDCG@10 now | recorded | paper | Hit@10 now | recorded | paper |")
print("| --- | --- | --- | --- | --- | --- | --- |")
for method in TAGS:
    n = mean([now[method, b][0] for b in (1, 2, 3)]); h = mean([now[method, b][1] for b in (1, 2, 3)])
    rn = mean([RECORDED[method][b][0] for b in (1, 2, 3)]); rh = mean([RECORDED[method][b][1] for b in (1, 2, 3)])
    print(f"| {method} | {n:.4f} | {rn:.4f} | {PAPER[method][0]:.4f} | {h:.4f} | {rh:.4f} | {PAPER[method][1]:.4f} |")
print("\nPESO - single LoRA, NDCG@10 (blocks 1-3):",
      f"now {mean([now['peso', b][0] - now['lora', b][0] for b in (1, 2, 3)]):+.4f};",
      f"recorded {mean([RECORDED['peso'][b][0] - RECORDED['lora'][b][0] for b in (1, 2, 3)]):+.4f};",
      f"paper {PAPER['peso'][0] - PAPER['lora'][0]:+.4f}")